# Data Pipeline

## Purpose

This notebook loads and reviews the raw Spotify/Kaggle datasets for **VibeML: A Personalized Spotify Playlist Generation Agent**.

The goal of this notebook is to inspect the available datasets, identify the columns needed for recommendation logic, and create a cleaned/processed dataset that can be used by the agent.

## Data Sources

This notebook uses the following raw tables from the Databricks catalog:

- `main.default.spotify_tracks_raw`  
  Main song dataset with audio features such as energy, valence, tempo, and danceability.

- `main.default.spotify_data_clean_raw`  
  Supporting Spotify global music dataset with track and artist metadata.

- `main.default.track_data_final_raw`  
  Supporting Spotify global music dataset with artist popularity, followers, genres, and track metadata.

## MVP Note

For the MVP version, the project will use the Kaggle Spotify datasets as the main data foundation. Ashley’s Spotify account and live Spotify API access will be treated as stretch goals for now.

## Data Setup Note

This notebook expects the raw Kaggle datasets to already be uploaded into Databricks Catalog as the following tables:

- `main.default.spotify_tracks_raw`
- `main.default.spotify_data_clean_raw`
- `main.default.track_data_final_raw`

IPORTATNT: These tables were created from the Kaggle Spotify datasets listed in the project proposal. _If another team member runs this notebook in a different workspace or cannot access these tables, they will need to download the Kaggle CSV files and upload them into Databricks using the same table names._

In [0]:
# Load raw tables from Databricks catalog

spotify_tracks_df = spark.table("main.default.spotify_tracks_raw")
spotify_data_clean_df = spark.table("main.default.spotify_data_clean_raw")
track_data_final_df = spark.table("main.default.track_data_final_raw")

In [0]:
# Check row counts for each raw table

print("Spotify Tracks rows:", spotify_tracks_df.count())
print("Spotify Data Clean rows:", spotify_data_clean_df.count())
print("Track Data Final rows:", track_data_final_df.count())

In [0]:
# Preview each dataset

display(spotify_tracks_df.limit(5))
display(spotify_data_clean_df.limit(5))
display(track_data_final_df.limit(5))

In [0]:
# Check column names and data types

spotify_tracks_df.printSchema()
spotify_data_clean_df.printSchema()
track_data_final_df.printSchema()

## Initial Dataset Review

The row counts and schemas show that `spotify_tracks_raw` will be the main dataset for the MVP recommendation logic because it has 114,000 tracks and includes audio features such as danceability, energy, valence, tempo, acousticness, speechiness, and instrumentalness. This is the dataset the agent will use to match mood/vibe to songs.

The two global music dataset tables, `spotify_data_clean_raw` and `track_data_final_raw`, are smaller supporting datasets. These tables include artist and album metadata such as artist popularity, artist followers, artist genres, album release date, and album type. These fields can support trend filtering, popularity filtering, and release-era filtering.

For the MVP, the data pipeline will focus on preparing `spotify_tracks_raw` as the main processed recommendation table. The global music tables will be reviewed as supporting data for enrichment if the fields can be joined or used separately by the agent tools.

In [0]:
# List columns for each dataset

print("spotify_tracks_raw columns:")
print(spotify_tracks_df.columns)

print("\nspotify_data_clean_raw columns:")
print(spotify_data_clean_df.columns)

print("\ntrack_data_final_raw columns:")
print(track_data_final_df.columns)

## Column Review

After reviewing the columns, `spotify_tracks_raw` appears to be the strongest dataset for the MVP recommendation logic. It contains the main song information, genre, popularity, and audio feature columns needed to match songs to a user's mood, activity, or vibe. These features include danceability, energy, valence, tempo, acousticness, instrumentalness, speechiness, and liveness.

The `spotify_data_clean_raw` and `track_data_final_raw` tables are supporting datasets from the Spotify Global Music Dataset. These tables do not include the audio feature columns needed for vibe matching, but they do include useful artist and album metadata, such as artist popularity, followers, genres, release date, and album type. These fields may be useful later for trend filtering, popularity filtering, or release-era filtering.

For the MVP, the first processed table will be built from `spotify_tracks_raw`. The global music tables will be kept as supporting data for enrichment if needed.

## Missing Value Check

Before creating the processed recommendation table, we checked the important columns from `spotify_tracks_raw` to see whether any values were missing. These columns are needed for the agent’s song search and vibe-matching logic.

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

# Important columns for recommendation logic
important_track_columns = [
    "track_id",
    "track_name",
    "artists",
    "track_genre",
    "popularity",
    "danceability",
    "energy",
    "valence",
    "tempo",
    "acousticness",
    "instrumentalness",
    "speechiness",
    "duration_ms",
    "explicit"
]

missing_counts_df = spotify_tracks_df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in important_track_columns
])

display(missing_counts_df)

## Missing Value Review

The missing value check shows that the main recommendation dataset is mostly complete. Only `track_name` and `artists` have one missing value each. Since these columns are needed for readable playlist recommendations, rows missing those values will be removed from the processed table.

The audio feature columns needed for recommendation logic, including danceability, energy, valence, tempo, acousticness, instrumentalness, and speechiness, have no missing values. This makes `spotify_tracks_raw` a strong foundation for the MVP recommendation dataset.

In [0]:
from pyspark.sql.functions import col

# Create the first processed recommendation table for the MVP

vibeml_tracks_processed_df = (
    spotify_tracks_df
    .select(
        "track_id",
        "track_name",
        "artists",
        "album_name",
        "track_genre",
        "popularity",
        "danceability",
        "energy",
        "valence",
        "tempo",
        "acousticness",
        "instrumentalness",
        "speechiness",
        "liveness",
        "duration_ms",
        "explicit"
    )
    .dropna(subset=[
        "track_id",
        "track_name",
        "artists",
        "track_genre",
        "danceability",
        "energy",
        "valence",
        "tempo"
    ])
    .dropDuplicates(["track_id"])
)

print("Processed track rows:", vibeml_tracks_processed_df.count())

display(vibeml_tracks_processed_df.limit(10))

In [0]:
# Check how many unique track IDs exist in the raw dataset

raw_row_count = spotify_tracks_df.count()
unique_track_count = spotify_tracks_df.select("track_id").distinct().count()

print("Raw row count:", raw_row_count)
print("Unique track_id count:", unique_track_count)
print("Duplicate rows based on track_id:", raw_row_count - unique_track_count)

## Processed Table Review

The first processed MVP recommendation table was created from `spotify_tracks_raw`. The raw dataset contains 114,000 rows, with 89,741 unique `track_id` values. This means 24,259 rows were duplicates based on `track_id`.

The final processed table contains 89,740 rows. Most of the row reduction came from removing duplicate `track_id` values. One additional row was removed because it was missing a required recommendation field.

This table keeps the columns needed for the agent’s song search and vibe-matching logic, including track name, artist, genre, popularity, and audio features such as danceability, energy, valence, tempo, acousticness, instrumentalness, and speechiness.

This processed table will be used as the main recommendation dataset for the MVP version of VibeML.

In [0]:
# Save the processed MVP recommendation table as a Delta table

vibeml_tracks_processed_df.write.mode("overwrite").format("delta").saveAsTable(
    "main.default.vibeml_tracks_processed"
)

In [0]:
# Confirm the processed Delta table was saved correctly

vibeml_tracks_processed_check_df = spark.table("main.default.vibeml_tracks_processed")

print("Saved processed table rows:", vibeml_tracks_processed_check_df.count())

display(vibeml_tracks_processed_check_df.limit(10))

## Saved Processed Table

The processed MVP recommendation dataset was saved as the Delta table `main.default.vibeml_tracks_processed`.

The saved table contains 89,740 rows and includes the main track metadata and audio feature columns needed for VibeML’s song search and vibe-matching logic. This table is now ready to be used by the agent’s recommendation tools.

## Basic Filtering Test

To confirm that the processed table can support the future song search tool, we created a simple filter for high-energy and high-danceability songs. This is not the final recommendation logic yet, but it shows that the dataset can be queried based on audio features.

In [0]:
from pyspark.sql.functions import col

# Basic test: find songs that are high energy and high danceability

high_energy_dance_df = (
    vibeml_tracks_processed_check_df
    .filter(col("energy") >= 0.75)
    .filter(col("danceability") >= 0.70)
    .select(
        "track_name",
        "artists",
        "track_genre",
        "popularity",
        "danceability",
        "energy",
        "valence",
        "tempo"
    )
    .orderBy(col("popularity").desc())
)

print("High-energy / high-danceability song count:", high_energy_dance_df.count())

display(high_energy_dance_df.limit(20))

In [0]:
# Basic test: find calmer songs with lower energy and higher acousticness

calm_playlist_test_df = (
    vibeml_tracks_processed_check_df
    .filter(col("energy") <= 0.45)
    .filter(col("acousticness") >= 0.50)
    .select(
        "track_name",
        "artists",
        "track_genre",
        "popularity",
        "danceability",
        "energy",
        "valence",
        "tempo",
        "acousticness"
    )
    .orderBy(col("popularity").desc())
)

print("Calm/acoustic song count:", calm_playlist_test_df.count())

display(calm_playlist_test_df.limit(20))

## Filtering Test Review

The filtering tests show that the processed table can support basic song search logic using audio feature ranges. The high-energy and high-danceability test returned 7,311 songs, which shows that the dataset can support upbeat playlist requests. The calm/acoustic test returned 17,360 songs, which shows that the dataset can also support lower-energy playlist requests.

These tests are not the final recommendation logic yet, but they confirm that the processed table can be filtered by features such as `energy`, `danceability`, `acousticness`, `valence`, `tempo`, genre, and popularity. This supports the future song search tool because the agent can translate a user request into feature targets and then query `main.default.vibeml_tracks_processed` for matching songs.

For the next stage, this filtering logic can be turned into a reusable function that accepts feature ranges as inputs and returns matching song recommendations.